In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

import matplotlib.pyplot as plt
from os import listdir
from os.path import isfile, join
import pandas as pd
import os
import numpy as np
import umap
from sklearn.datasets import fetch_openml
import seaborn as sns
from sklearn import preprocessing
from sklearn import cluster
from sklearn import manifold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

/home/jakob/.pyenv/versions/3.13.12/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1779892619.842884   28215 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779892619.877970   28215 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779892620.825779   28215 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to float

In [2]:
#Next, we´ll get the low-res photos imported(doesnt take long. for my computer: 7.5s)

#Creating a list for the photos with the corresponding dates and image names as well(crude way of being able to see which are outliers.
#Needs to be done better, really. If someone has the time, feel free to find a better way of doing things)
images_loaded = []

#Next, a list for the metadata i want(for now, at least. there´s certainly going to be a better way of doing this later):
metadata_image_clustering = []


#Folder with folders of images:
base_folder = r"../data/scrandle_padded_low_res_data"
#Meta data folders(yes, a bit inefficient):
meta_folder = r"../data/grand_scraper_folder/scrandle_data"


#Looping over all dates in the listed directory/folder:
for date in sorted(os.listdir(base_folder)):
    
    #Finding a new date folder to check
    date_path = os.path.join(base_folder, date)
    meta_date_path = os.path.join(meta_folder, date)
    #Checking if the date path exists
    if not os.path.isdir(date_path):
        continue

    print(f"Processing {date}")

    #Looping over all images in the given folder:
    for image_file_name in ['0_left.webp', '0_right.webp', '1_left.webp', '1_right.webp', '2_left.webp', '2_right.webp', '3_left.webp', '3_right.webp', '4_left.webp', '4_right.webp', '5_left.webp', '5_right.webp', '6_left.webp', '6_right.webp', '7_left.webp', '7_right.webp', '8_left.webp', '8_right.webp', '9_left.webp', '9_right.webp']:
        image_path = os.path.join(date_path, image_file_name)
        images_loaded.append({
                "date": date,
                "image_name": image_file_name,
                "RGB": plt.imread(image_path, format="webp"),
            })
    
    #Finding the other metadata:
    data_path = os.path.join(meta_date_path, "meta.csv")
    meta_intermediate = pd.read_csv(data_path)
    metadata_image_clustering.append({
                "price": meta_intermediate["price"],
                "rating": meta_intermediate["rating"],
                "year": meta_intermediate["year"],
                "date": meta_intermediate["date"],
                "image_name": meta_intermediate["image_name"],
            })

Processing 2025-04-20
Processing 2025-04-21
Processing 2025-04-22
Processing 2025-04-23
Processing 2025-04-24
Processing 2025-04-25
Processing 2025-04-26
Processing 2025-04-27
Processing 2025-04-28
Processing 2025-04-29
Processing 2025-04-30
Processing 2025-05-01
Processing 2025-05-02
Processing 2025-05-03
Processing 2025-05-04
Processing 2025-05-05
Processing 2025-05-06
Processing 2025-05-07
Processing 2025-05-08
Processing 2025-05-09
Processing 2025-05-10
Processing 2025-05-11
Processing 2025-05-12
Processing 2025-05-13
Processing 2025-05-14
Processing 2025-05-15
Processing 2025-05-16
Processing 2025-05-17
Processing 2025-05-18
Processing 2025-05-19
Processing 2025-05-20
Processing 2025-05-21
Processing 2025-05-22
Processing 2025-05-23
Processing 2025-05-24
Processing 2025-05-25
Processing 2025-05-26
Processing 2025-05-27
Processing 2025-05-28
Processing 2025-05-29
Processing 2025-05-30
Processing 2025-05-31
Processing 2025-06-01
Processing 2025-06-02
Processing 2025-06-03
Processing

In [10]:
ratings = []
for i in range(len(metadata_image_clustering)):
    ratings += metadata_image_clustering[i]["rating"].to_list()
ratings = torch.tensor(ratings) / 100


In [11]:
ratings = (
    pd.concat([m["rating"] for m in metadata_image_clustering], ignore_index=True)
    .to_numpy(dtype=np.float32)
)
ratings = torch.from_numpy(ratings) / 100.0

images = np.stack([img["RGB"] for img in images_loaded]).astype(np.float32)
images = torch.from_numpy(images) / 255.0

images = images.permute(0,3,1,2)

In [12]:
images.shape

torch.Size([7900, 3, 100, 85])

In [25]:
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1) 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")
model = model.to(device)

class ImageRatingDataset(Dataset):
    def __init__(self, ratings, images):
        if (len(ratings) != len(images)):
            raise ValueError("Error: 'ratings' and 'images is not of equal length!")
        self.ratings = ratings
        self.images = images

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        image = self.images[idx]
        rating = self.ratings[idx]

        return image, rating


dataset = ImageRatingDataset(ratings, images)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


model.train()

for epoch in range(5):
    total_loss = 0.0
    
    for images_loaded, ratings_loaded in loader:
        images_temp = images_loaded.to(device)
        ratings_temp = ratings_loaded.to(device)

        preds = model(images_temp)
        loss = criterion(preds, ratings_temp)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1}: loss={total_loss/len(loader):.4f}")


model.eval()
with torch.no_grad():
    
    img = images[0]
    pred = model(images).item()
    pred = max(0, min(100, pred)) 
    print("Predicted rating:", pred)
    

Using Device: cpu


KeyboardInterrupt: 

In [7]:
images.shape

torch.Size([16, 3, 100, 85])

In [ ]:
images

TypeError: 'builtin_function_or_method' object is not subscriptable